# Overton Window — Baseline Exploration

**SI 630: Natural Language Processing · Winter 2026**  
**Paris Heard · School of Information, University of Michigan**

---

This notebook documents the **initial exploratory pipeline** built on the Babel Briefings dataset before the final dataset was identified. It is included for methodological transparency: understanding what was tried, what it revealed, and why it was replaced is part of the research record.

### What It Covers
1. Loading and inspecting the Babel Briefings corpus (54 multilingual JSON files)
2. Filtering to US English headlines
3. Implementing the TF-IDF centroid distance framework as proof-of-concept
4. Evaluating results and identifying dataset limitations
5. Documenting the decision to switch to the US Multi-Outlet News Headlines dataset

### Why Babel Briefings Was Replaced
The Babel Briefings corpus covers only **August 2020 – November 2021** (16 monthly bins), which is far too short to measure the long-term discourse drift the research question requires. The US English subset contains ~92,443 headlines, compared to 5,804,010 in the final dataset. Results on Babel Briefings provided proof-of-concept validation (the November 2021 Omicron spike was cleanly detected), but the temporal range made it unsuitable as a primary corpus.

**The final analysis uses:** `dess-mannheim/US_Multi_Outlet_News_Headlines2001_2024` (see `02_main_analysis.ipynb`)

## 0. Imports & Setup

In [ ]:
import random
import numpy as np
random.seed(42)
np.random.seed(42)

In [ ]:
import os
import re
import json
import glob
import pandas as pd
from pathlib import Path
from datetime import datetime

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Preprocessing parameters (carried forward to final analysis)
MIN_WORDS = 4

print('Imports OK')

## 1. Data Loading — Babel Briefings

Babel Briefings (Leeb & Schölkopf, 2024) is a multilingual news headline corpus distributed as 54 JSON files organized by country/language code. Each file contains a list of article objects with fields including `title`, `publishedAt`, `language`, and `source-name`.

**Data directory:** `./babel_briefings/` (not included in repo — see `data/README.md`)

In [ ]:
DATA_DIR = '../data/babel_briefings/'

# recursively find all JSON files
json_files = glob.glob(os.path.join(DATA_DIR, '**', '*.json'), recursive=True)
print(f'Found {len(json_files)} JSON files')
print('Example paths:', json_files[:3])

In [ ]:
# inspect schema of first file
with open(json_files[0], 'r') as f:
    sample = json.load(f)

# handle both list-of-articles and dict-wrapped formats
if isinstance(sample, list):
    sample_articles = sample
elif isinstance(sample, dict):
    # common wrapper keys include 'articles', 'data', 'items', etc.
    sample_articles = next(iter(sample.values()))

print(f'Records in first file: {len(sample_articles)}')
print(f'Field names: {list(sample_articles[0].keys())}')
print('\nSample record:')
print(json.dumps({k: sample_articles[0][k] for k in ['title', 'publishedAt', 'language', 'source-name']}, indent=2))

In [ ]:
# load all files, filter to English headlines (partially redundant if only viewing US headlines, but ensures consistency)
all_records = []

for path in json_files:
    with open(path, 'r') as f:
        data = json.load(f)

    for item in data:
        if item.get('language') == 'en' and item.get('title'):
            all_records.append({
                'title':       item['title'],
                'date':        item.get('publishedAt') or item.get('collectedAt'),
                'source':      item.get('source-name', 'unknown'),
                'language':    item.get('language'),
            })

df_babel = pd.DataFrame(all_records)

print(f'Total English records: {len(df_babel):,}')
print(f'Unique sources: {df_babel["source"].nunique()}')
print(df_babel.head())

## 2. Preprocessing

Same pipeline applied in the final analysis: lowercase, strip URLs, keep letters/digits/hyphens, enforce minimum word count.

In [ ]:
TITLE_FIELD    = 'title'        # headline text field
DATE_FIELD     = 'publishedAt'  # ISO date string
LANGUAGE_FIELD = 'language'    # language code (e.g. 'en')
SOURCE_FIELD   = 'source-name'  # publication name (may be nested)
CONTENT_FIELD  = 'content'      # full article text (if available)

def parse_date(date_str):
    """
    Parse ISO 8601 dates from Babel Briefings to YYYY-MM.
    """
    if not date_str:
        return pd.NaT

    # strip trailing timezone info variants
    date_str = re.sub(r'(\.\d+)?[Z\+].*$', '', str(date_str))

    for fmt in ('%Y-%m-%dT%H:%M:%S', '%Y-%m-%d %H:%M:%S', '%Y-%m-%d'):
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue

    return pd.NaT

def preprocess(text):
    """
    Normalize headline: lowercase, strip URLs and punctuation, collapse whitespace.
    """
    text = str(text).lower()
    text = re.sub(r'http\S+',        '', text)
    text = re.sub(r'[^a-z0-9\s\-]', '', text)
    text = re.sub(r'\s+',            ' ', text).strip()

    return text

def is_valid(text, min_words=MIN_WORDS):
    """
    Return True if headline meets minimum word count after preprocessing.
    """
    return len(str(text).split()) >= min_words

def load_article(article):
    """
    Extract relevant fields from a single article dict.
    Handles missing fields and nested source dicts.
    """
    title = article.get(TITLE_FIELD, '')
    title = preprocess(title)

    if not is_valid(title):
        return

    date  = parse_date(article.get(DATE_FIELD))
    lang  = article.get(LANGUAGE_FIELD, '')
    content = article.get(CONTENT_FIELD, '')

    # source may be a nested dict: {'name': 'Reuters'}
    src   = article.get(SOURCE_FIELD, '')

    if isinstance(src, dict):
        src = src.get('name', '')

    return {'title': title, 'date': date, 'language': lang, 'source': src, 'content': content}

print("Configurations set. Ready to load and preprocess articles.")

In [ ]:
# regex to extract US path (review data/README.md for link to data source with additional country codes)
US_PATH_REGEX = re.compile(r'.*us.*\.json$', re.IGNORECASE)
US_FILES = [file for file in json_files if US_PATH_REGEX.match(file)]

#  load and preprocess all articles, filter to US headlines
records = []
errors = 0

for file in US_FILES:
    try:
        with open(file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            articles = data if isinstance(data, list) else next(iter(data.values()))
            for art in articles:
                records.append(load_article(art))
    except Exception as e:
        errors += 1
        print(f"Error processing file {file}: {e}")

# filter out none values from records
valid_records = [record for record in records if record is not None]

df = pd.DataFrame(valid_records)
df['date'] = pd.to_datetime(df['date'])
print(f'Total records loaded : {len(df):,}')
print(f'Date range           : {df.date.min().date()} → {df.date.max().date()}')
df.head(10)

## 3. TF-IDF Proof-of-Concept

The first 12 months are used as a baseline; the remaining 4 as the comparison window.

> **Note:** This split is purely exploratory — the Babel Briefings date range (Aug 2020 – Nov 2021) does not map onto a theoretically motivated baseline/comparison division. The short window was the primary reason for switching datasets.

In [ ]:
df['year_month'] = df['date'].dt.to_period('M')
months_sorted = df['year_month'].sort_values().unique()
print('Months in dataset:', ', '.join(str(m) for m in months_sorted))

In [ ]:
# distribution of headlines per month
monthly_counts = df.groupby('year_month').size().reset_index(name='count')

plt.figure(figsize=(12, 6))
plt.plot(monthly_counts['year_month'].astype(str), monthly_counts['count'], marker='o')
plt.title('Monthly Headline Counts (US)')
plt.xlabel('Month')
plt.ylabel('Number of Headlines')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# cutoff for minimum headline frequency to include in analysis
MIN_HEADLINE_FREQ = 30

# aggregate headlines by month
monthly_headlines = df.groupby('year_month')['title'].apply(list).reset_index()

# filter to months with sufficient headline volume
monthly_headlines['headline_count'] = monthly_headlines['title'].apply(len)
monthly_headlines = monthly_headlines[monthly_headlines['headline_count'] >= MIN_HEADLINE_FREQ]
print(f'Months with ≥{MIN_HEADLINE_FREQ} headlines: {len(monthly_headlines)}')
monthly_headlines.head()

In [ ]:
# baseline months are those with at least MIN_HEADLINE_FREQ headlines, sorted by date, and taking the first N months
BASELINE_MONTHS = 12
COMPARISON_MONTHS = 4

baseline_months = monthly_headlines.sort_values('year_month').head(BASELINE_MONTHS)['year_month'].tolist()
comparison_months = monthly_headlines.sort_values('year_month').tail(COMPARISON_MONTHS)['year_month'].tolist()

print(f'Baseline months  : {baseline_months[0]} – {baseline_months[-1]}')
print(f'Comparison months: {comparison_months[0]} – {comparison_months[-1]}')

In [ ]:
df_base = df[df['year_month'].isin(baseline_months)]
df_comp = df[df['year_month'].isin(comparison_months)]

print(f'Baseline  : {len(df_base):,} headlines')
print(f'Comparison: {len(df_comp):,} headlines')

In [ ]:
# fit vectorizer on baseline only (same design decision as final analysis)
MAX_FEATURES = 10_000    # vocabulary cap -- increase if your compute allows
NGRAM_RANGE  = (1, 2)    # unigrams + bigrams
MIN_DF       = 5         # ignore super rare terms

vectorizer = TfidfVectorizer(
    max_features = MAX_FEATURES,
    ngram_range  = NGRAM_RANGE,
    min_df       = MIN_DF,
    sublinear_tf = True,
    stop_words   = 'english',    # log-scale term frequency weighting, remove common words to focus on distinctive terms
)
X_base = vectorizer.fit_transform(df_base['title'])
X_comp = vectorizer.transform(df_comp['title'])

centroid_baseline = np.asarray(X_base.mean(axis=0)).flatten()

print(f'Vocabulary size: {len(vectorizer.get_feature_names_out()):,}')
print(f'Baseline matrix: {X_base.shape}')
print(f'Baseline centroid shape: {centroid_baseline.shape}')

In [ ]:
# monthly cosine distance from baseline centroid
# note: this is a very simple approach to quantifying topical shifts over time, but can reveal interesting aggregate patterns
# and capture broad changes in the overall "center of mass" of linguistic content, even if the specific terms driving those shifts may require deeper analysis to fully understand

df_base = df_base.reset_index(drop=True)
df_comp = df_comp.reset_index(drop=True)
records = []

for corpus_label, df, X in [('baseline', df_base, X_base), ('comparison', df_comp, X_comp)]:
    for month, group in df.groupby('year_month'):
        idx  = group.index.tolist()
        mat  = X[idx]
        ctrd = np.asarray(mat.mean(axis=0)).flatten()
        sim  = cosine_similarity(ctrd.reshape(1,-1), centroid_baseline.reshape(1,-1))[0,0]
        dist = 1.0 - float(sim)

        sims_to_centroid = cosine_similarity(mat, ctrd.reshape(1,-1)).flatten()
        spread = float(np.std(sims_to_centroid))

        records.append({
            'month':              month,
            'cosine_dist':        dist,
            'intra_month_spread': spread,
            'corpus':             corpus_label,
            'n':                  len(idx)
        })

df_results = pd.DataFrame(records).sort_values('month').reset_index(drop=True)
print(df_results.to_string(index=False))

## 4. Results & Limitations

The Babel Briefings proof-of-concept confirmed that the centroid distance framework is sensitive to genuine discourse events. A Mann-Whitney U test comparing monthly cosine distances to the baseline centroid yielded a U-statistic of 4.0 (p = 0.007), confirming that comparison period months are significantly more distant from the baseline centroid than baseline period months, despite only 16 monthly bins total. The small U-statistic indicates that nearly all comparison months rank higher in cosine distance than baseline months, consistent with the directional drift predicted by the Overton Window hypothesis. The November 2021 Omicron spike (distance = 0.142) was cleanly detected. However, two critical limitations ruled it out as the primary corpus:

1. **Short Temporal Range**: 16 monthly bins faces difficulty supporting a study of long-term discourse drift. The research question requires at larger corpus, preferably a decade of baseline data.
2. **Sparse Pre-2016 Coverage**: The Overton Window shift hypothesis concerns a stable baseline reference period. The pre-2016 political period, which is markedly noted as more stable and less divisive, is not available.
3. **Headline Artifacts**: As noted in the top TF-IDF terms, occassionally the outlet names will be included in the headline text, which is a unique artifact of this dataset. This could be mitigated with additional cleaning and filtering, but presents an additional limitation.

These limitations motivated the switch to the US Multi-Outlet News Headlines dataset, which covers 2001–2024 across five major US outlets.

In [ ]:
# visualize proof-of-concept results
all_months = sorted(df_results['month'].unique())
m2x = {m: i for i, m in enumerate(all_months)}

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('Babel Briefings — Proof-of-Concept: Cosine Distance from Baseline Centroid', fontsize=12)

for corpus, color in [('baseline', 'steelblue'), ('comparison', 'darkorange')]:
    sub = df_results[df_results['corpus'] == corpus]
    x   = [m2x[m] for m in sub['month']]
    axes[0].plot(x, sub['cosine_dist'],        color=color, marker='o', linewidth=1.5, label=corpus.capitalize())
    axes[1].plot(x, sub['intra_month_spread'], color=color, marker='o', linewidth=1.5, label=corpus.capitalize())

axes[0].set_ylabel('Cosine Distance from Baseline Centroid')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

axes[1].set_ylabel('Intra-Month Spread')
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

axes[1].set_xticks(range(len(all_months)))
axes[1].set_xticklabels(all_months, rotation=45, ha='right', fontsize=8)

Path('outputs/figures').mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig('outputs/figures/babel_proof_of_concept.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: babel_proof_of_concept.png')

In [ ]:
# statistical significance testing (mann whitney test to evaluate 
# whether comparison months are significantly less similar to baseline centroids
# than the baseline months themselves, which would suggest a meaningful shift in topical content over time)

baseline_dists    = df_results[df_results['corpus'] == 'baseline']['cosine_dist'].values
comparison_dists  = df_results[df_results['corpus'] == 'comparison']['cosine_dist'].values
u_stat, p_value = mannwhitneyu(baseline_dists, comparison_dists, alternative='less')

print('Statistical Significance Testing (Comparison vs. Baseline):')
print('\nMann-Whitney U Test (Comparison < Baseline):')
print(f'U-statistic: {u_stat:.4f}')
print(f'P-value    : {p_value:.4e}')

In [ ]:
# summary statistics

print("=== Dataset Statistics ===")
print(f'Total English headlines: {len(df):,}')
print(f'Unique sources       : {df["source"].nunique()}')
print(f'Date range           : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Months in dataset    : {len(df["year_month"].unique())}')
print(f'Months in baseline   : {len(baseline_months)}')
print(f'Months in comparison : {len(comparison_months)}')

print("\n=== Baseline vs Comparison ===")
print(f'Baseline headlines  : {len(df_base):,}')
print(f'Comparison headlines: {len(df_comp):,}')

print('\n=== Cosine Similarity to Baseline Centroid ===')
print(df_results[['month', 'corpus', 'cosine_dist', 'intra_month_spread']].to_string(index=False))

print('\nTop 20 TF-IDF Terms in Baseline Corpus:')
feature_names = vectorizer.get_feature_names_out()
tfidf_means = np.asarray(X_base.mean(axis=0)).flatten()
top_indices = np.argsort(tfidf_means)[::-1][:20]
for idx in top_indices:
    print(f'{feature_names[idx]}: {tfidf_means[idx]:.4f}')

## 5. Dataset Search & Decision Log

The following datasets were evaluated before selecting the final corpus:

| Dataset | Reason Rejected/Accepted |
|---|---|
| Stanford OVAL CC-News (600M articles, 2016–2024) | No North America filter; no pre-2016 coverage |
| 3DLNews2 (US local news, 1995–2024) | 7GB Globus download; local-only coverage |
| HEADLINES semantic similarity (Dell/Harvard, 1920–1989) | Date range mismatch |
| Factiva RTF exports (via U-M library) | 100-article-per-download limit made bulk acquisition infeasible |
| **US Multi-Outlet News Headlines 2001–2024** | **Selected: temporal range, acceptable exploratory outlet diversity, US-specificity, HuggingFace streaming** |

**→ Continue to `02_main_analysis.ipynb` for the full pipeline.**